# Capitolo 4 - Studio sperimentale e valutazione dei modelli di forecasting

**Analisi dei modelli di forecasting per serie temporali**

Candidato: Maral Noori 
Universita degli Studi di Modena e Reggio Emilio - A.A. 2025/2026

---

> **Prima di eseguire:** caricare i file `kole 2024.csv` e `kole 2025.csv` nella sessione Colab.
> (File gia estratti dall'API OpenF1 e salvati sul PC)

In [ ]:
# ============================================================
# [4.1] Import delle librerie (library imports)
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OrdinalEncoder
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from pmdarima import auto_arima

np.random.seed(42)  # riproducibilita (reproducibility)
print('Librerie caricate con successo.')

In [ ]:
# ============================================================
# [4.2] Configurazione del dataset (data configuration)
# ============================================================

# Nomi dei Gran Premi come compaiono nel dataset
GP_NAMES = [
    'Bahrain Grand Prix',
    'Emilia Romagna Grand Prix',
    'Austrian Grand Prix',
    'Italian Grand Prix',
    'Singapore Grand Prix'
]

# Variabile target e predittori - Tabella 4.2
TARGET = 'lap_duration'
PREDICTORS = [
    'compound', 'tyre_age', 'pit_stop',
    'air_temperature', 'track_temperature',
    'humidity', 'wind_speed', 'rainfall'
]

# Ordine delle mescole per codifica ordinale
COMPOUND_ORDER = ['SOFT', 'MEDIUM', 'HARD']

## 4.4 - Analisi preliminare del dataset
> Caricare i file CSV nella sessione Colab prima di eseguire questa cella.
> I dataset sono gia stati estratti dall'API OpenF1 e salvati sul PC.

In [ ]:
# ============================================================
# [4.4] Caricamento e verifica qualita dei dataset
# ============================================================

# Carichiamo i file CSV salvati sul PC (upload su Colab)
df_2024 = pd.read_csv('kole 2024.csv')
df_2025 = pd.read_csv('kole 2025.csv')

# Dimensioni dei dataset
print(f'Training set (2024): {df_2024.shape[0]} osservazioni, {df_2024.shape[1]} variabili')
print(f'Test set     (2025): {df_2025.shape[0]} osservazioni, {df_2025.shape[1]} variabili')
print()

# Valori mancanti nella variabile target
print(f'Mancanti in lap_duration 2024: {df_2024[TARGET].isna().sum()}')
print(f'Mancanti in lap_duration 2025: {df_2025[TARGET].isna().sum()}')
print()

# Mancanti per colonna (solo quelle con valori mancanti)
missing = df_2024.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print('Mancanti nel training set (duration_sector_1 ha 2, pit_duration e\'normale):')
    print(missing)
else:
    print('Nessun valore mancante nel training set.')

In [ ]:
# ============================================================
# [4.5] Analisi esplorativa - Tabella 4.1
# ============================================================

stat_count = df_2024[TARGET].count()
stat_mean  = df_2024[TARGET].mean()
stat_std   = df_2024[TARGET].std()
stat_min   = df_2024[TARGET].min()
stat_q25   = df_2024[TARGET].quantile(0.25)
stat_med   = df_2024[TARGET].quantile(0.50)
stat_q75   = df_2024[TARGET].quantile(0.75)
stat_max   = df_2024[TARGET].max()

print('=== Tabella 4.1 - Statistiche descrittive della variabile target (2024) ===')
print(f'{"Statistica":<25} {"Valore":>10}')
print('-' * 37)
print(f'{"Count":<25} {stat_count:>10.1f}')
print(f'{"Mean":<25} {stat_mean:>10.3f}')
print(f'{"Standard deviation":<25} {stat_std:>10.3f}')
print(f'{"Minimum":<25} {stat_min:>10.3f}')
print(f'{"25th percentile":<25} {stat_q25:>10.3f}')
print(f'{"Median":<25} {stat_med:>10.3f}')
print(f'{"75th percentile":<25} {stat_q75:>10.3f}')
print(f'{"Maximum":<25} {stat_max:>10.3f}')

In [ ]:
# ============================================================
# [4.5] Istogramma della distribuzione - Figura 4.1
# ============================================================

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_2024[TARGET], bins=30, edgecolor='black', alpha=0.7, color='#1f77b4')
ax.set_title('Figura 4.1 - Distribuzione della variabile lap duration nella stagione 2024',
             fontsize=10)
ax.set_xlabel('Lap duration (s)')
ax.set_ylabel('Frequenza')
plt.tight_layout()
plt.savefig('figura_4_1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# [4.6] Preparazione dati per il forecasting - Tabella 4.2
# ============================================================

print('=== Tabella 4.2 - Configurazione del dataset ===')
print(f'{"Elemento":<25} {"Descrizione":<55}')
print('-' * 80)
print(f'{"Training set":<25} {"Stagione 2024 (" + str(df_2024.shape[0]) + " osservazioni)":<55}')
print(f'{"Test set":<25} {"Stagione 2025 (" + str(df_2025.shape[0]) + " osservazioni)":<55}')
print(f'{"Variabile target":<25} {"Tempo sul giro (lap_duration)":<55}')
print(f'{"Predittori":<25} {"Compound, tyre age, pit stop, air temperature, track temperature, humidity, wind speed, rainfall":<55}')

## Funzione di utilita - Metriche di errore
> Calcola MAE, RMSE, MAPE e MASE. Usata per tutti i modelli.

In [ ]:
# ============================================================
# Funzione metriche di errore (error metrics utility)
# ============================================================

def compute_metrics(y_true, y_pred, y_train):
    """
    Calcola le 4 metriche usate nella tesi:
    MAE  - Errore assoluto medio
    RMSE - Radice dell'errore quadratico medio
    MAPE - Errore percentuale assoluto medio (%)
    MASE - Errore assoluto medio scalato
    """
    y_true  = np.array(y_true, dtype=float)
    y_pred  = np.array(y_pred, dtype=float)
    y_train = np.array(y_train, dtype=float)

    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    naive_mae = np.mean(np.abs(np.diff(y_train)))
    mase = mae / naive_mae if naive_mae > 0 else np.inf

    return {'MAE': round(mae, 3), 'RMSE': round(rmse, 3),
            'MAPE': round(mape, 3), 'MASE': round(mase, 3)}

## 4.7.1 - Valutazione del modello Nave
> Il modello Nave prevede che il prossimo giro sia uguale all'ultimo osservato.
> Le prestazioni sono valutate su ciascun Gran Premio.

In [ ]:
# ============================================================
# [4.7.1] Modello Nave - Tabella 4.3 + Figura 4.2
# ============================================================

results_naive = []

fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=False)

for i, gp in enumerate(GP_NAMES):
    train = df_2024[df_2024['grand_prix'] == gp][TARGET].values
    test  = df_2025[df_2025['grand_prix'] == gp][TARGET].values

    # Nave: previsione costante = ultimo valore del training
    naive_pred = np.full(len(test), train[-1])

    # Metriche (Tabella 4.3)
    m = compute_metrics(test, naive_pred, train)
    m['Grand Prix'] = gp
    results_naive.append(m)

    # Plot: Figura 4.2
    ax = axes[i]
    ax.plot(range(len(test)), test, label='Osservato', color='black', linewidth=1)
    ax.plot(range(len(test)), naive_pred, label='Nave', color='#1f77b4', linestyle='--')
    ax.set_title(gp.replace(' Grand Prix', ' GP'), fontsize=9)
    ax.set_xlabel('Giro')
    if i == 0:
        ax.set_ylabel('Lap duration (s)')
    ax.legend(fontsize=7)

plt.suptitle('Figura 4.2 - Confronto tra valori osservati e previsioni Nave', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('figura_4_2_naive.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabella 4.3
df_n = pd.DataFrame(results_naive).set_index('Grand Prix')
print('=== Tabella 4.3 - Prestazioni del modello Nave ===')
print(df_n.to_string())

## 4.7.2 - Valutazione del modello Drift
> Il modello Drift proietta la variazione media tra primo e ultimo valore
> della serie di addestramento.

In [ ]:
# ============================================================
# [4.7.2] Modello Drift - Tabella 4.4 + Figura 4.3
# ============================================================

results_drift = []

fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=False)

for i, gp in enumerate(GP_NAMES):
    train = df_2024[df_2024['grand_prix'] == gp][TARGET].values
    test  = df_2025[df_2025['grand_prix'] == gp][TARGET].values

    # Drift: variazione media = (y_T - y_1) / (T - 1)
    slope = (train[-1] - train[0]) / max(len(train) - 1, 1)
    drift_pred = train[-1] + slope * np.arange(1, len(test) + 1)

    # Metriche (Tabella 4.4)
    m = compute_metrics(test, drift_pred, train)
    m['Grand Prix'] = gp
    results_drift.append(m)

    # Plot: Figura 4.3
    ax = axes[i]
    ax.plot(range(len(test)), test, label='Osservato', color='black', linewidth=1)
    ax.plot(range(len(test)), drift_pred, label='Drift', color='#ff7f0e', linestyle='--')
    ax.set_title(gp.replace(' Grand Prix', ' GP'), fontsize=9)
    ax.set_xlabel('Giro')
    if i == 0:
        ax.set_ylabel('Lap duration (s)')
    ax.legend(fontsize=7)

plt.suptitle('Figura 4.3 - Confronto tra valori osservati e previsioni Drift', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('figura_4_3_drift.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabella 4.4
df_d = pd.DataFrame(results_drift).set_index('Grand Prix')
print('=== Tabella 4.4 - Prestazioni del modello Drift ===')
print(df_d.to_string())

## 4.7.3 - Scelta del benchmark di riferimento
> Il modello Nave e stato scelto come riferimento principale per il confronto,
> perche in grado di utilizzare l'informazione temporale contenuta nella serie.
> Il metodo Mean non e stato incluso (previsioni costanti pari alla media).
> Il Seasonal Nave non e stato applicato (ciascun GP e trattato come serie indipendente).
> Le prestazioni dei modelli successivi saranno valutate rispetto al Nave.

## 4.7.4 - Regressione Lineare
> A differenza dei benchmark, la regressione lineare integra le informazioni
> relative agli pneumatici e alle condizioni ambientali.

In [ ]:
# ============================================================
# [4.7.4] Regressione Lineare - Tabella 4.5
# ============================================================

def encode_compound(df):
    """Codifica la variabile compound in valori numerici ordinali."""
    enc = OrdinalEncoder(categories=[COMPOUND_ORDER],
                         handle_unknown='use_encoded_value',
                         unknown_value=-1)
    return enc.fit_transform(df[['compound']]).flatten()


def prepare_features(train_df, test_df):
    """
    Prepara le variabili predittive per la regressione.
    Codifica compound e seleziona i predittori.
    """
    tr = train_df.copy()
    te = test_df.copy()
    tr['compound_enc'] = encode_compound(tr)
    te['compound_enc'] = encode_compound(te)
    feat = ['compound_enc'] + [c for c in PREDICTORS if c != 'compound']
    return tr[feat].fillna(0), tr[TARGET].values, te[feat].fillna(0), te[TARGET].values


results_lr = []

for gp in GP_NAMES:
    train_df = df_2024[df_2024['grand_prix'] == gp]
    test_df  = df_2025[df_2025['grand_prix'] == gp]
    X_tr, y_tr, X_te, y_te = prepare_features(train_df, test_df)

    lr = LinearRegression()
    lr.fit(X_tr, y_tr)
    lr_pred = lr.predict(X_te)

    m = compute_metrics(y_te, lr_pred, y_tr)
    m['Grand Prix'] = gp
    results_lr.append(m)

# Tabella 4.5
df_lr = pd.DataFrame(results_lr).set_index('Grand Prix')
print('=== Tabella 4.5 - Metriche del modello di regressione lineare ===')
df_lr_display = df_lr.rename(columns={'MAPE': 'MAPE (%)'})
print(df_lr_display.to_string())

In [ ]:
# ============================================================
# [4.7.4] Analisi anomalia Bahrain GP 2025 - Tabella 4.6
# ============================================================

bah = df_2025[df_2025['grand_prix'] == 'Bahrain Grand Prix'].copy()

# Giri 27-37 del Bahrain GP 2025
tab_46 = bah.loc[
    bah['lap_number'].between(27, 37),
    ['lap_number', 'lap_duration', 'pit_stop', 'pit_duration',
     'compound', 'tyre_age', 'stint_number']
]

# Rinominiamo le colonne come nella tesi
tab_46_display = tab_46.rename(columns={
    'lap_number': 'Lap_number',
    'lap_duration': 'Lap_duration (s)',
    'pit_stop': 'pit_stop',
    'pit_duration': 'Pit_duration (s)',
    'compound': 'compound',
    'tyre_age': 'Tyre_age',
    'stint_number': 'Stint_number'
})

print('=== Tabella 4.6 - Variabili esplicative tra i giri 27-37 del Bahrain GP 2025 ===')
print(tab_46_display.to_string(index=False))
print()
print('-> Giri 33-35: Safety Car (Tsunoda-Sainz). Lap 32 pit stop + neutralizzazione.')

In [ ]:
# ============================================================
# [4.7.4] Regressione Lineare DOPO interpolazione Bahrain - Tabella 4.7
# ============================================================

bah_clean = df_2025[df_2025['grand_prix'] == 'Bahrain Grand Prix'].copy()

# I valori anomali dei giri 33, 34, 35 vengono sostituiti
# mediante interpolazione lineare
sc_mask = bah_clean['lap_number'].isin([33, 34, 35])
bah_clean.loc[sc_mask, TARGET] = np.nan
bah_clean[TARGET] = bah_clean[TARGET].interpolate(method='linear')

# Stesso modello di prima, addestrato sui dati del Bahrain 2024
X_tr_b, y_tr_b, X_te_b, y_te_b = prepare_features(
    df_2024[df_2024['grand_prix'] == 'Bahrain Grand Prix'], bah_clean)
lr_b = LinearRegression().fit(X_tr_b, y_tr_b)
pred_b = lr_b.predict(X_te_b)
m_b = compute_metrics(y_te_b, pred_b, y_tr_b)

print('=== Tabella 4.7 - Reg. lineare Bahrain DOPO interpolazione valori anomali ===')
print(f'{"Grand Prix":<25} {"MAE":>8} {"RMSE":>8} {"MAPE (%)":>10} {"MASE":>8}')
print('-' * 60)
print(f'{"Bahrain Grand Prix":<25} {m_b["MAE"]:>8.3f} {m_b["RMSE"]:>8.3f} '
      f'{m_b["MAPE"]:>10.3f} {m_b["MASE"]:>8.3f}')
print()
print('-> Le metriche si riducono sensibilmente rispetto ai valori precedenti.')

## 4.7.5 - Modello ARIMA
> La previsione e effettuata utilizzando esclusivamente la serie storica di lap_duration.
> AutoARIMA seleziona automaticamente l'ordine (p,d,q) che minimizza l'AIC.

In [ ]:
# ============================================================
# [4.7.5] Modello ARIMA con selezione automatica - Tabella 4.8
# ============================================================

results_arima = []

for gp in GP_NAMES:
    train = df_2024[df_2024['grand_prix'] == gp][TARGET].values
    test  = df_2025[df_2025['grand_prix'] == gp][TARGET].values

    # AutoARIMA seleziona (p,d,q) che minimizza l'AIC
    auto = auto_arima(train, seasonal=False, stepwise=True,
                      suppress_warnings=True)
    fitted = auto.fit(train)
    arima_pred = fitted.predict(n_periods=len(test))

    m = compute_metrics(test, arima_pred, train)
    m['Grand Prix'] = gp
    m['ARIMA Order'] = str(auto.order)
    m['AIC'] = round(auto.aic(), 2)
    results_arima.append(m)

# Tabella 4.8
df_arima = pd.DataFrame(results_arima).set_index('Grand Prix')
print('=== Tabella 4.8 - Ordini ARIMA selezionati e prestazioni previsionali ===')
cols_48 = ['ARIMA Order', 'AIC', 'MAE', 'RMSE', 'MAPE', 'MASE']
df_48 = df_arima[cols_48].rename(columns={'MAPE': 'MAPE (%)'})
print(df_48.to_string())

In [ ]:
# ============================================================
# [4.7.5] Test di Ljung-Box sui residui ARIMA - Tabella 4.9
# ============================================================

print('=== Tabella 4.9 - Risultati del test di Ljung-Box sui residui ARIMA ===')
print(f'{"Grand Prix":<30} {"p-value":>10}')
print('-' * 42)

for gp in GP_NAMES:
    train = df_2024[df_2024['grand_prix'] == gp][TARGET].values
    auto = auto_arima(train, seasonal=False, stepwise=True,
                      suppress_warnings=True)
    fitted = auto.fit(train)
    residuals = fitted.resid

    lb = acorr_ljungbox(residuals, lags=10, return_df=True)
    pval = lb['lb_pvalue'].min()
    ok = 'Adeguato' if pval > 0.05 else 'Autocorrelazione'
    print(f'{gp:<30} {pval:>10.4f}  {ok}')

print()
print('-> Tutti i p-value > 0.05: residui compatibili con rumore bianco.')

## 4.7.6 - Regressione Dinamica
> Per ciascun GP e stimata prima una regressione lineare. Se i residui mostrano
> autocorrelazione (Ljung-Box p <= 0.05), si aggiunge una componente ARIMA.

In [ ]:
# ============================================================
# [4.7.6] Regressione Dinamica - Tabella 4.10
# ============================================================

results_dyn = []

for gp in GP_NAMES:
    train_df = df_2024[df_2024['grand_prix'] == gp]
    test_df  = df_2025[df_2025['grand_prix'] == gp]
    X_tr, y_tr, X_te, y_te = prepare_features(train_df, test_df)

    # Step 1: regressione lineare
    lr = LinearRegression().fit(X_tr, y_tr)
    resid_train = y_tr - lr.predict(X_tr)

    # Step 2: Ljung-Box sui residui della regressione
    lb = acorr_ljungbox(resid_train, lags=10, return_df=True)
    min_pval = lb['lb_pvalue'].min()

    arima_order = None
    if min_pval <= 0.05:
        # Autocorrelazione significativa -> aggiungiamo componente ARIMA
        auto = auto_arima(resid_train, seasonal=False, suppress_warnings=True)
        arima_order = auto.order
        model = ARIMA(y_tr, order=arima_order, exog=X_tr).fit()
        dyn_pred = model.forecast(steps=len(y_te), exog=X_te)
    else:
        # Nessuna autocorrelazione -> solo regressione
        arima_order = (0, 0, 0)
        dyn_pred = lr.predict(X_te)

    m = compute_metrics(y_te, dyn_pred, y_tr)
    m['Grand Prix'] = gp
    m['ARIMA'] = str(arima_order)
    results_dyn.append(m)

# Tabella 4.10
df_dyn = pd.DataFrame(results_dyn).set_index('Grand Prix')
print('=== Tabella 4.10 - Prestazioni del modello di regressione dinamica ===')
df_410 = df_dyn[['MAE', 'RMSE', 'MAPE', 'MASE', 'ARIMA']].rename(columns={'MAPE': 'MAPE (%)'})
print(df_410.to_string())

In [ ]:
# ============================================================
# [4.7.6] Test di Ljung-Box sulle innovazioni - Tabella 4.11
# ============================================================

print('=== Tabella 4.11 - Ljung-Box sulle innovazioni della Reg. Dinamica ===')
print(f'{"Grand Prix":<30} {"p-value":>12}')
print('-' * 44)

for gp in GP_NAMES:
    train_df = df_2024[df_2024['grand_prix'] == gp]
    test_df  = df_2025[df_2025['grand_prix'] == gp]
    X_tr, y_tr, X_te, y_te = prepare_features(train_df, test_df)

    lr = LinearRegression().fit(X_tr, y_tr)
    resid_train = y_tr - lr.predict(X_tr)
    lb = acorr_ljungbox(resid_train, lags=10, return_df=True)

    if lb['lb_pvalue'].min() <= 0.05:
        auto = auto_arima(resid_train, seasonal=False, suppress_warnings=True)
        model = ARIMA(y_tr, order=auto.order, exog=X_tr).fit()
        innovations = y_te - model.forecast(steps=len(y_te), exog=X_te)
    else:
        innovations = y_te - lr.predict(X_te)

    lb_innov = acorr_ljungbox(innovations, lags=10, return_df=True)
    pval = lb_innov['lb_pvalue'].min()
    print(f'{gp:<30} {pval:>12.2e}')

print()
print('-> Tutti i p-value < 0.05: le innovazioni presentano autocorrelazione residua.')

## 4.8 - Confronto tra i modelli
> Il MASE rappresenta l'indicatore piu adatto per il confronto, perche fornisce
> una misura dell'errore indipendente dalla scala dei dati.

In [ ]:
# ============================================================
# [4.8] Confronto modelli - Tabella 4.12 (MASE comparison)
# ============================================================

comparison = pd.DataFrame({
    'Naive':                  pd.DataFrame(results_naive).set_index('Grand Prix')['MASE'],
    'Drift':                  pd.DataFrame(results_drift).set_index('Grand Prix')['MASE'],
    'Regressione lineare':    pd.DataFrame(results_lr).set_index('Grand Prix')['MASE'],
    'ARIMA':                  pd.DataFrame(results_arima).set_index('Grand Prix')['MASE'],
    'Regressione dinamica':   pd.DataFrame(results_dyn).set_index('Grand Prix')['MASE'],
})

print('=== Tabella 4.12 - Confronto dei valori di MASE ===')
print(comparison.to_string())
print()

print('Modello migliore per Gran Premio (MASE piu basso):')
for gp in comparison.index:
    row = comparison.loc[gp]
    best = row.idxmin()
    print(f'  {gp:<30} -> {best} (MASE = {row[best]})')

In [ ]:
# ============================================================
# [4.8] Heatmap del confronto MASE
# ============================================================

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(comparison.values, cmap='YlOrRd_r', aspect='auto')
ax.set_xticks(range(len(comparison.columns)))
ax.set_xticklabels(comparison.columns, fontsize=9, rotation=15, ha='right')
ax.set_yticks(range(len(comparison.index)))
short_names = [g.replace(' Grand Prix', ' GP') for g in comparison.index]
ax.set_yticklabels(short_names, fontsize=9)

for i in range(len(comparison.index)):
    for j in range(len(comparison.columns)):
        v = comparison.values[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9,
                color='white' if v > 10 else 'black')

plt.colorbar(im, ax=ax, label='MASE (basso = migliore)')
ax.set_title('Confronto MASE tra modelli di forecasting', fontsize=11)
plt.tight_layout()
plt.savefig('heatmap_mase.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.9 - Considerazioni finali sullo studio sperimentale
> L'analisi evidenzia come il comportamento dei modelli dipenda dalle caratteristiche
> della serie temporale e dal contesto in cui viene applicata la previsione.
> La scelta del modello piu appropriato richiede una valutazione basata sulle
> prestazioni osservate, piuttosto che sull'adozione di approcci progressivamente
> piu complessi.

In [ ]:
# ============================================================
# Riepilogo finale delle prestazioni
# ============================================================

all_models = {
    'Naive':              pd.DataFrame(results_naive).set_index('Grand Prix'),
    'Drift':              pd.DataFrame(results_drift).set_index('Grand Prix'),
    'Reg. Lineare':       pd.DataFrame(results_lr).set_index('Grand Prix'),
    'ARIMA':              pd.DataFrame(results_arima).set_index('Grand Prix'),
    'Reg. Dinamica':      pd.DataFrame(results_dyn).set_index('Grand Prix'),
}

rows = []
for name, df in all_models.items():
    rows.append({
        'Modello':     name,
        'MAE mediano':  df['MAE'].median(),
        'RMSE mediano': df['RMSE'].median(),
        'MASE mediano': df['MASE'].median(),
    })

summary = pd.DataFrame(rows).set_index('Modello')
print('=== RIEPILOGO RISULTATI (median across GPs) ===')
print(summary.to_string())
print()
print('-> Il modello con MASE mediano piu basso e il migliore in media.')